In [68]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

In [69]:
from dotenv import load_dotenv
load_dotenv()

True

In [70]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")

# Load the unified preprocessed TSV file with image hashes
annotations_file = os.path.join(data_dir, "preprocessed_annotations.tsv")

In [71]:
df = pd.read_csv(annotations_file, sep='\t')

In [72]:
df.shape

(18082, 27)

In [73]:
df.head()

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,...,img_width,img_height,img_aspect_ratio,img_hash_str,text_dup,image_dup,mm_info,mm_human,cleaned_text_bert,cleaned_text_bertweet
0,917791044158185473,917791044158185473_0,informative,1.0000,informative,0.6766,other_relevant_information,1.0000,other_relevant_information,0.6766,...,800,450,1.777778,86c476bb39c2b16c,False,False,informative,other_relevant_information,Wildfires raging through Northern California a...,Wildfires raging through Northern California a...
1,917791130590183424,917791130590183424_0,informative,1.0000,informative,0.6667,infrastructure_and_utility_damage,1.0000,affected_individuals,0.6667,...,1200,677,1.772526,f28e9aa98b96a730,False,False,informative,conflict,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California ht...
2,917791291823591425,917791291823591425_0,informative,0.6813,informative,1.0000,other_relevant_information,0.6813,infrastructure_and_utility_damage,1.0000,...,640,480,1.333333,a9bdb18391838bba,False,False,informative,infrastructure_and_utility_damage,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
3,917791291823591425,917791291823591425_1,informative,0.6813,not_informative,1.0000,other_relevant_information,0.6813,not_humanitarian,1.0000,...,1200,900,1.333333,b582566fab45cac8,True,False,informative,other_relevant_information,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
4,917792092100988929,917792092100988929_0,informative,0.6727,informative,0.6612,other_relevant_information,0.6727,infrastructure_and_utility_damage,0.6612,...,600,400,1.500000,86826be7394ed43a,False,False,informative,infrastructure_and_utility_damage,California's raging wildfires as you've never ...,California's raging wildfires as you've never ...


In [74]:
# Parameters
RANDOM_SEED = 42
TRAIN_PERC = 0.70
VAL_PERC = 0.15
TEST_PERC = 0.15

## Unimodal text classification splits 

In [75]:
# Remove text duplicates
text_df = df[df['text_dup'] == False]

# Select relevant columns
cols = ['tweet_text', 'text_info', 'text_human', 'cleaned_text_bert', 'cleaned_text_bertweet']
text_df = text_df[cols]

In [76]:
text_df.shape

(16058, 5)

In [77]:
text_df.head()

,tweet_text,text_info,text_human,cleaned_text_bert,cleaned_text_bertweet
0,RT @Gizmodo: Wildfires raging through Northern...,informative,other_relevant_information,Wildfires raging through Northern California a...,Wildfires raging through Northern California a...
1,PHOTOS: Deadly wildfires rage in California ht...,informative,infrastructure_and_utility_damage,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California ht...
2,RT @Cal_OES: PLS SHARE: We're capturing wildfi...,informative,other_relevant_information,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
4,RT @TIME: California's raging wildfires as you...,informative,other_relevant_information,California's raging wildfires as you've never ...,California's raging wildfires as you've never ...
5,Wildfires Threaten California's First Legal Ca...,informative,other_relevant_information,Wildfires Threaten California's First Legal Ca...,Wildfires Threaten California's First Legal Ca...


In [78]:
# Stratified split
# First split train vs temp (val+test)
text_df_train, text_df_temp = train_test_split(
    text_df,
    test_size=(1-TRAIN_PERC),
    stratify=text_df['text_human'],
    random_state=RANDOM_SEED
)

# Then split temp into val and test
val_size = VAL_PERC / (VAL_PERC + TEST_PERC)  # fraction of temp to go to val
text_df_val, text_df_test = train_test_split(
    text_df_temp,
    test_size=1-val_size,
    stratify=text_df_temp['text_human'],
    random_state=RANDOM_SEED
)

In [79]:
print(f"Train: {len(text_df_train)}, Val: {len(text_df_val)}, Test: {len(text_df_test)}")

def print_class_distribution(df, label_col):
    counts = df[label_col].value_counts(dropna=False)
    percentages = df[label_col].value_counts(normalize=True, dropna=False) * 100
    print(pd.DataFrame({'count': counts, 'percent': percentages.round(2)}))

print(f"\nTrain set class distribution:")
print_class_distribution(text_df_train, 'text_info')
print_class_distribution(text_df_train, 'text_human')
print(f"\nValidation set class distribution:")
print_class_distribution(text_df_val, 'text_info')
print_class_distribution(text_df_val, 'text_human')
print(f"\nTest set class distribution:")
print_class_distribution(text_df_test, 'text_info')
print_class_distribution(text_df_test, 'text_human')

Train: 11240, Val: 2409, Test: 2409

Train set class distribution:
                 count  percent
text_info                      
informative       8064    71.74
not_informative   3176    28.26
                                        count  percent
text_human                                            
other_relevant_information               4161    37.02
not_humanitarian                         3192    28.40
rescue_volunteering_or_donation_effort   2304    20.50
infrastructure_and_utility_damage         885     7.87
affected_individuals                      698     6.21

Validation set class distribution:
                 count  percent
text_info                      
informative       1728    71.73
not_informative    681    28.27
                                        count  percent
text_human                                            
other_relevant_information                891    36.99
not_humanitarian                          684    28.39
rescue_volunteering_or_donation_effo

In [80]:
# Save splits
train_save_path = os.path.join(datasplits_dir, "text_train.tsv")
val_save_path = os.path.join(datasplits_dir, "text_val.tsv")
test_save_path = os.path.join(datasplits_dir, "text_test.tsv")

text_df_train.to_csv(train_save_path, sep='\t', index=False)
text_df_val.to_csv(val_save_path, sep='\t', index=False)
text_df_test.to_csv(test_save_path, sep='\t', index=False)

## Unimodal image classification splits

In [81]:
# Remove duplicated images
img_df = df[df['image_dup'] == False]

# Select relevant columns
cols = ['img_hash_str', 'image_path', 'image_info', 'image_human']
img_df = img_df[cols]

In [82]:
img_df.shape

(17354, 4)

In [83]:
img_df.head()

,img_hash_str,image_path,image_info,image_human
0,86c476bb39c2b16c,data_image/california_wildfires/10_10_2017/917...,informative,other_relevant_information
1,f28e9aa98b96a730,data_image/california_wildfires/10_10_2017/917...,informative,affected_individuals
2,a9bdb18391838bba,data_image/california_wildfires/10_10_2017/917...,informative,infrastructure_and_utility_damage
3,b582566fab45cac8,data_image/california_wildfires/10_10_2017/917...,not_informative,not_humanitarian
4,86826be7394ed43a,data_image/california_wildfires/10_10_2017/917...,informative,infrastructure_and_utility_damage


In [84]:
# Douvle-check successful removal of duplicates
unique_img_hashes = img_df['img_hash_str'].nunique()
print(f"Number of unique img_hash_str values: {unique_img_hashes}")

Number of unique img_hash_str values: 17354


In [85]:
# Stratified split
# First split train vs temp (val+test)
img_df_train, img_df_temp = train_test_split(
    img_df,
    test_size=(1-TRAIN_PERC),
    stratify=img_df['image_human'],
    random_state=RANDOM_SEED
)

# Then split temp into val and test
val_size = VAL_PERC / (VAL_PERC + TEST_PERC)  # fraction of temp to go to val
img_df_val, img_df_test = train_test_split(
    img_df_temp,
    test_size=1-val_size,
    stratify=img_df_temp['image_human'],
    random_state=RANDOM_SEED
)

In [86]:
print(f"Train: {len(img_df_train)}, Val: {len(img_df_val)}, Test: {len(img_df_test)}")

print(f"\nTrain set class distribution:")
print_class_distribution(img_df_train, 'image_info')
print_class_distribution(img_df_train, 'image_human')
print(f"\nValidation set class distribution:")
print_class_distribution(img_df_val, 'image_info')
print_class_distribution(img_df_val, 'image_human')
print(f"\nTest set class distribution:")
print_class_distribution(img_df_test, 'image_info')
print_class_distribution(img_df_test, 'image_human')

Train: 12147, Val: 2603, Test: 2604

Train set class distribution:
                 count  percent
image_info                     
informative       6251    51.46
not_informative   5896    48.54
                                        count  percent
image_human                                           
not_humanitarian                         5913    48.68
infrastructure_and_utility_damage        2595    21.36
other_relevant_information               1683    13.86
rescue_volunteering_or_donation_effort   1498    12.33
affected_individuals                      458     3.77

Validation set class distribution:
                 count  percent
image_info                     
informative       1338     51.4
not_informative   1265     48.6
                                        count  percent
image_human                                           
not_humanitarian                         1267    48.67
infrastructure_and_utility_damage         556    21.36
other_relevant_information          

In [87]:
# Save splits
train_save_path = os.path.join(datasplits_dir, "img_train.tsv")
val_save_path = os.path.join(datasplits_dir, "img_val.tsv")
test_save_path = os.path.join(datasplits_dir, "img_test.tsv")

img_df_train.to_csv(train_save_path, sep='\t', index=False)
img_df_val.to_csv(val_save_path, sep='\t', index=False)
img_df_test.to_csv(test_save_path, sep='\t', index=False)

## Multimodal classification splits

In [88]:
# TODO